# Cuantización 4/8-bit con `bitsandbytes`

Cargamos el mismo modelo (`NousResearch/Llama-3.2-1B`) en 3 configuraciones y comparamos VRAM ocupada y calidad del output:

- **full precision** (bfloat16): baseline.
- **8-bit**: cada peso pasa a 1 byte.
- **4-bit** (`nf4`): cada peso pasa a medio byte — mismo esquema que usa QLoRA (módulo 7).

Con GPU de 8GB (RTX 4060), un modelo de 1B ya entra cómodo en las 3 configs — la comparación de memoria es la que importa, no si entra o no.

In [16]:
import logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)  # silencia el warning "MatMul8bitLt..." repetido por cada capa

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "NousResearch/Llama-3.2-1B"
PROMPT = "El futuro de la inteligencia artificial es"

assert torch.cuda.is_available(), "Sin GPU CUDA disponible"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

## Función helper

Carga el modelo con la config dada, mide VRAM real ocupada (`torch.cuda.memory_allocated`) y genera una respuesta corta. Reseteamos el modelo y limpiamos la cache de CUDA entre cargas para que la medición de cada config no arrastre memoria de la anterior.

In [17]:
def cargar_y_medir(nombre, quantization_config=None):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        quantization_config=quantization_config,
        device_map="cuda",
    )

    vram_gb = torch.cuda.memory_allocated() / 1e9

    inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")
    salida = model.generate(**inputs, max_new_tokens=30, do_sample=False)
    texto = tokenizer.decode(salida[0], skip_special_tokens=True)

    del model
    torch.cuda.empty_cache()

    print(f"--- {nombre} ---")
    print(f"VRAM ocupada: {vram_gb:.2f} GB")
    print(f"Output: {texto}\n")
    return vram_gb, texto

In [18]:
vram_full, texto_full = cargar_y_medir("Full precision (bfloat16)")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

--- Full precision (bfloat16) ---
VRAM ocupada: 2.48 GB
Output: El futuro de la inteligencia artificial es en la inteligencia humana
El futuro de la inteligencia artificial es en la inteligencia humana
El futuro de la inteligencia artificial es



In [19]:
config_8bit = BitsAndBytesConfig(load_in_8bit=True)
vram_8bit, texto_8bit = cargar_y_medir("8-bit", config_8bit)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

--- 8-bit ---
VRAM ocupada: 1.51 GB
Output: El futuro de la inteligencia artificial es la inteligencia artificial de la inteligencia artificial
El futuro de la inteligencia artificial es la inteligencia artificial de la inteligencia artificial
El futuro



In [20]:
config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
vram_4bit, texto_4bit = cargar_y_medir("4-bit (nf4)", config_4bit)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

--- 4-bit (nf4) ---
VRAM ocupada: 1.08 GB
Output: El futuro de la inteligencia artificial es el de la inteligencia humana
El futuro de la inteligencia artificial es el de la inteligencia humana
El futuro de la inteligencia



In [21]:
print(f"Full precision: {vram_full:.2f} GB")
print(f"8-bit:          {vram_8bit:.2f} GB  ({vram_full/vram_8bit:.1f}x menos)")
print(f"4-bit:          {vram_4bit:.2f} GB  ({vram_full/vram_4bit:.1f}x menos)")

Full precision: 2.48 GB
8-bit:          1.51 GB  (1.6x menos)
4-bit:          1.08 GB  (2.3x menos)
